In [ ]:
%%capture
%cd ~/repos/PredUCE/
%load_ext autoreload
%autoreload 2

In [ ]:
from dotenv import load_dotenv

import polars as pl

from make_clinical_dataset.epic.combine import merge_closest_measurements
from make_clinical_dataset.epr.prep import Splitter
from make_clinical_dataset.shared.constants import ROOT_DIR

load_dotenv()

EMBEDDING_SECTIONS = [
    "active_symptoms",
    "recent_complications",
    "healthcare_utilization",
    "functional_status",
    "medication_risks",
    "psychosocial_risks",
    "clinical_uncertainty",
    "acuity_assessment",
]

EMB_MODEL = "PubMedBERT"
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"
DATA_PATH = f'{DATA_DIR}/processed/treatment_centered_data.parquet'
DATE_PATH = f'{DATA_DIR}/processed/treatment_centered_dates.parquet'
NOTE_PATH = f'{DATA_DIR}/interim/subsets/clinic_visits_prior_to_treatment/notes.parquet'

TEXT_PATH = f'{DATA_DIR}/interim/embedding/ed_risk_summary.parquet'
EMB_PATH = f'{DATA_DIR}/interim/embedding/{EMB_MODEL}'

In [ ]:
# Load data
tabular_df = pl.read_parquet(DATA_PATH)
note_df = pl.read_parquet(NOTE_PATH, columns=["mrn", "clinic_date", "note_id"])
text_df = pl.read_parquet(TEXT_PATH, columns=["note_id"] + [f"{section}_text_id" for section in EMBEDDING_SECTIONS])

# Get the closest clinical note prior to assessment date within the lookback window
note_df = note_df.rename({"clinic_date": "prev_clinic_date", "note_id": "prev_note_id"})
df = merge_closest_measurements(tabular_df, note_df, "assessment_date", "prev_clinic_date", merge_individually=False, time_window=(-30,-1))
del tabular_df, note_df

# Include the text section ids
df = df.join(text_df, left_on="prev_note_id", right_on="note_id", how="left")

In [ ]:
# Pre-process the data
# keep only the first treatment of a given week
df = df.group_by_dynamic("assessment_date", every="7d", group_by="mrn").agg(pl.all().first())

# split the data - create development (EPR) and test (EPIC) set
# split_date: str = "2022-01-01"
# splitter = Splitter()
# dev_data, test_data = splitter.temporal_split(
#     df,
#     split_date=split_date,
#     visit_col="assessment_date",
#     exclude_after_split=False,
# )